# FlashSpec Quickstart

This notebook demonstrates speculative sampling and bandit selection
using toy models — no GPU or real model weights required.

**Run time**: < 30 seconds on any machine.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
from flashspec.utils.device import set_seed
from flashspec.utils.config import FlashSpecConfig, BanditConfig, SamplingConfig
set_seed(42)
print('FlashSpec imported OK')

## 1. Configuration

In [ ]:
config = FlashSpecConfig(
    device='cpu',
    dtype='float32',
    drafter_name='toy',
    target_name='toy',
    max_new_tokens=64,
    bandit=BanditConfig(n_arms=1, strategy='ucb1'),
    sampling=SamplingConfig(gamma=4, temperature=1.0),
)
print(config.model_dump_json(indent=2))

## 2. One speculative step

In [ ]:
from flashspec.sampling.rejection import rejection_sample

BATCH, GAMMA, VOCAB = 2, 4, 1000
set_seed(7)
draft_lp   = torch.randn(BATCH, GAMMA, VOCAB).log_softmax(-1)
target_lp  = torch.randn(BATCH, GAMMA, VOCAB).log_softmax(-1)
draft_ids  = torch.randint(0, VOCAB, (BATCH, GAMMA))
ctx        = torch.randint(0, VOCAB, (BATCH, 16))

accepted_ids, first_rejection, alpha = rejection_sample(
    input_ids=ctx,
    draft_logprobs=draft_lp,
    target_logprobs=target_lp,
    draft_token_ids=draft_ids,
    gamma=GAMMA,
)
print(f'accepted_ids shape : {accepted_ids.shape}')
print(f'first_rejection    : {first_rejection.tolist()}')
print(f'alpha (mean)       : {alpha:.3f}')

## 3. UCB1 bandit selection demo

In [ ]:
import numpy as np
from flashspec.bandit import UCB1Selector

selector = UCB1Selector(n_arms=2, window_size=100)
rng = np.random.default_rng(0)
true_rates = [0.4, 0.75]

print(f'{"Round":>6} | {"Arm":>4} | {"Arm-0 pulls":>12} | {"Arm-1 pulls":>12}')
print('-' * 44)
for t in range(30):
    arm = selector.select()
    accepted = int(rng.random() < true_rates[arm])
    selector.update(arm, accepted)
    if (t + 1) % 5 == 0:
        print(f'{t+1:>6} | {arm:>4} | {selector._arms[0].n_pulls:>12} | {selector._arms[1].n_pulls:>12}')

print(f'\nArm-0 acceptance rate: {selector._arms[0].mean_accept_rate:.3f}')
print(f'Arm-1 acceptance rate: {selector._arms[1].mean_accept_rate:.3f}')

## 4. Metrics snapshot

In [ ]:
from flashspec.metrics import AcceptanceTracker, LatencyTracker
import time

acc_tracker = AcceptanceTracker(gamma=4)
lat_tracker = LatencyTracker(window=100)

for _ in range(50):
    lat_tracker.start()
    time.sleep(0.0005)   # simulate ~0.5 ms step
    lat_tracker.stop()
    acc_tracker.record(n_accepted=int(rng.integers(2, 5)))

print(f'Mean acceptance rate : {acc_tracker.mean_acceptance_rate:.3f}')
print(f'p50 latency          : {lat_tracker.p50_ms:.2f} ms')
print(f'p99 latency          : {lat_tracker.p99_ms:.2f} ms')